In [1]:
import numpy as np
import cv2
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\Images_Vision")

In [ ]:
orange=cv2.imread("orange.jpg")
orange=cv2.resize(orange,(512,512))
apple=cv2.imread("apple.jpg")
apple=cv2.resize(apple,(512,512))
apple_orange=np.hstack((orange[:,:256],apple[:,256:]))
cv2.imshow("Orange",orange)
cv2.imshow("Apple",apple)
cv2.imshow("Apple Orange",apple_orange)
cv2.waitKey(0)

cv2.destroyAllWindows()

In [38]:
apple=cv2.imread('apple.jpg')
apple=cv2.resize(apple,(512,512))
orange=cv2.imread('orange.jpg')
orange=cv2.resize(orange,(512,512))
apple_copy=apple.copy()
gp_apple=[apple_copy]
for i in range(6):
    apple_img=cv2.pyrDown(apple_copy)
    gp_apple.append(apple_img)

orange_copy=orange.copy()
gp_orange=[orange_copy]
for i in range(6):
    orange_img=cv2.pyrDown(orange_copy)
    gp_orange.append(orange_img)
last_apple=gp_apple[-1]
lp_apple=[last_apple]
for i,img in enumerate(gp_apple):
    print(i,img.shape)
for i in range(5,0,-1):
    extend=cv2.pyrUp(gp_apple[i])
    extend=cv2.resize(extend,(gp_apple[i-1].shape[1],gp_apple[i-1].shape[0]))
    laplacian=cv2.subtract(gp_apple[i-1],extend)
    lp_apple.append(laplacian)
     
last_orange=gp_orange[-1]
lp_orange=[last_orange]
for i in range(5,0,-1):
    extend=cv2.pyrUp(gp_orange[i])
    extend=cv2.resize(extend,(gp_orange[i-1].shape[1],gp_orange[i-1].shape[0]))
    laplacian=cv2.subtract(gp_orange[i-1],extend)
    lp_orange.append(laplacian)


apple_orange_pyramid=[]
for lap_apple,lap_orange in zip(lp_apple,lp_orange):
    rows,cols=lap_apple.shape[:2]
    laplacian=np.hstack((lap_apple[:, :cols//2], lap_orange[:, cols//2:]))
    apple_orange_pyramid.append(laplacian)
apple_orange_reconstruct=apple_orange_pyramid[0]
for i in range(1,6):
    apple_orange_reconstruct=cv2.pyrUp(apple_orange_reconstruct)
    apple_orange_reconstruct=cv2.resize(apple_orange_reconstruct,(apple_orange_pyramid[i].shape[1],apple_orange_pyramid[i].shape[0]))
    apple_orange_reconstruct=cv2.add(apple_orange_pyramid[i],apple_orange_reconstruct)
    
cv2.imshow('apple',apple)
cv2.imshow('orange',orange)
cv2.imshow('apple_orange_reconstruct',apple_orange_reconstruct)
cv2.waitKey(0)
cv2.destroyAllWindows()


0 (512, 512, 3)
1 (256, 256, 3)
2 (256, 256, 3)
3 (256, 256, 3)
4 (256, 256, 3)
5 (256, 256, 3)
6 (256, 256, 3)


# Simple project to apply concepts

In [1]:
import cv2
import numpy as np
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\Images_Vision")

# ============================================================
# 1. LOAD IMAGES
# ============================================================

landscape = cv2.imread("landscape.jpg")
sky = cv2.imread("sky.jpg")

if landscape is None:
    raise FileNotFoundError("Could not find landscape.jpg")

if sky is None:
    raise FileNotFoundError("Could not find sky.jpg")


# ============================================================
# 2. RESIZE IMAGES TO THE SAME SIZE
# ============================================================

height, width = landscape.shape[:2]

sky = cv2.resize(sky, (width, height))

print("Landscape:", landscape.shape)
print("Sky:", sky.shape)


# ============================================================
# 3. CREATE SKY MASK
# ============================================================
#
# White  = sky will be used
# Black  = landscape will be used
#
# IMPORTANT:
# Adjust sky_height according to your image.
#
# Example:
# 40% of image = sky region
#

sky_height = int(height * 0.40)

mask = np.zeros((height, width), dtype=np.float32)

mask[:sky_height, :] = 1.0


# ============================================================
# 4. BLUR THE MASK
# ============================================================
#
# This creates a smoother transition around the horizon.
#

mask = cv2.GaussianBlur(
    mask,
    (31, 31),
    0
)


# ============================================================
# 5. CREATE GAUSSIAN PYRAMIDS
# ============================================================

levels = 6

gp_landscape = [landscape.astype(np.float32)]
gp_sky = [sky.astype(np.float32)]
gp_mask = [mask]


for i in range(levels):

    gp_landscape.append(
        cv2.pyrDown(gp_landscape[-1])
    )

    gp_sky.append(
        cv2.pyrDown(gp_sky[-1])
    )

    gp_mask.append(
        cv2.pyrDown(gp_mask[-1])
    )


# ============================================================
# 6. CREATE LAPLACIAN PYRAMIDS
# ============================================================

lp_landscape = [gp_landscape[-1]]
lp_sky = [gp_sky[-1]]


for i in range(levels, 0, -1):

    # -------------------------
    # Landscape
    # -------------------------

    expanded_landscape = cv2.pyrUp(
        gp_landscape[i]
    )

    expanded_landscape = cv2.resize(
        expanded_landscape,
        (
            gp_landscape[i - 1].shape[1],
            gp_landscape[i - 1].shape[0]
        )
    )

    lap_landscape = cv2.subtract(
        gp_landscape[i - 1],
        expanded_landscape
    )

    lp_landscape.append(lap_landscape)


    # -------------------------
    # Sky
    # -------------------------

    expanded_sky = cv2.pyrUp(
        gp_sky[i]
    )

    expanded_sky = cv2.resize(
        expanded_sky,
        (
            gp_sky[i - 1].shape[1],
            gp_sky[i - 1].shape[0]
        )
    )

    lap_sky = cv2.subtract(
        gp_sky[i - 1],
        expanded_sky
    )

    lp_sky.append(lap_sky)


# ============================================================
# 7. BLEND LAPLACIAN PYRAMIDS
# ============================================================

blended_pyramid = []


for lap_landscape, lap_sky, mask_level in zip(
    lp_landscape,
    lp_sky,
    gp_mask[::-1]
):

    # Convert mask to 3 channels

    mask_3ch = cv2.merge([
        mask_level,
        mask_level,
        mask_level
    ])


    # Make sure dimensions match

    mask_3ch = cv2.resize(
        mask_3ch,
        (
            lap_landscape.shape[1],
            lap_landscape.shape[0]
        )
    )


    # Blend

    blended = (
        lap_sky * mask_3ch
        +
        lap_landscape * (1 - mask_3ch)
    )

    blended_pyramid.append(blended)


# ============================================================
# 8. RECONSTRUCT IMAGE
# ============================================================

result = blended_pyramid[0]


for i in range(1, len(blended_pyramid)):

    result = cv2.pyrUp(result)

    result = cv2.resize(
        result,
        (
            blended_pyramid[i].shape[1],
            blended_pyramid[i].shape[0]
        )
    )

    result = cv2.add(
        result,
        blended_pyramid[i]
    )


# ============================================================
# 9. CONVERT RESULT TO UINT8
# ============================================================

result = np.clip(result, 0, 255).astype(np.uint8)


# ============================================================
# 10. CREATE NORMAL MASK FOR DISPLAY
# ============================================================

display_mask = (mask * 255).astype(np.uint8)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

cv2.imshow(
    "Original Landscape",
    landscape
)

cv2.imshow(
    "Replacement Sky",
    sky
)

cv2.imshow(
    "Sky Mask",
    display_mask
)

cv2.imshow(
    "Final Blended Image",
    result
)


# ============================================================
# 12. SAVE RESULT
# ============================================================

cv2.imwrite(
    "sky_blended_result.jpg",
    result
)


print("\nProject completed!")
print("Result saved as: sky_blended_result.jpg")


# ============================================================
# 13. EXIT
# ============================================================

cv2.waitKey(0)
cv2.destroyAllWindows()

Landscape: (362, 551, 3)
Sky: (362, 551, 3)

Project completed!
Result saved as: sky_blended_result.jpg
